# From Abundance to Presence/Absence

GMWI2's model doesn't look at *how much* of a species is present — only
whether it's present at all. This notebook hand-builds that transformation
on a small teaching dataset shaped like a simplified species-abundance
table, the same "implement it by hand" style as the other course's
diversity-metrics notebook.

## 1. Load the data

`toy_species_abundance.csv` is **synthetic, illustrative data** — 80
samples, 12 species (6 loosely modeled on real health-associated genera
like *Faecalibacterium prausnitzii* and *Akkermansia muciniphila*, 6 on
real disease-associated genera like *Escherichia coli* and *Klebsiella
pneumoniae*), with a `health_status` label. It is **not** real patient
data and the exact numbers aren't from any published study — it's built
to have the same *shape* as real data so you can practice on it safely.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("toy_species_abundance.csv")
species = [c for c in df.columns if c not in ("sample_id", "health_status")]
print(df.shape, "->", df.health_status.value_counts().to_dict())
df.head(6)

## 2. Binarize: presence/absence at a threshold

The real GMWI2 pipeline treats a species as "present" if MetaPhlAn3
detects it above its default sensitivity threshold at all. We'll use a
simple, explicit rule: present if relative abundance > 0%.

In [ ]:
def binarize(df, species, threshold=0.0):
    pa = (df[species] > threshold).astype(int)
    return pa

presence_absence = binarize(df, species)
presence_absence.insert(0, "sample_id", df["sample_id"])
presence_absence.insert(1, "health_status", df["health_status"])
presence_absence.head(6)

### 🔧 YOUR TURN #1
Try `threshold=0.5` instead of `0.0` — i.e. only count a species as
present if it's above 0.5% relative abundance. **Predict first:** will
more or fewer species count as "present" overall? Then check by
comparing `presence_absence[species].sum().sum()` before and after.

In [ ]:
# Your code here

## 3. Does presence/absence alone separate the groups?

Before building any model (that's notebook 05), just look: do healthy and
non_healthy samples differ in *how many* of the 12 species they carry?

In [ ]:
presence_absence["n_species_present"] = presence_absence[species].sum(axis=1)
presence_absence.groupby("health_status")["n_species_present"].describe()[["mean", "std", "min", "max"]].round(2)

### EXPLAIN #1
*Look at the mean `n_species_present` for each group. Does simple species
*richness* (just a count, ignoring which species) look like it would
separate the two groups on its own? Why might GMWI2 need to know **which**
species are present, not just how many?*

> your answer here

## Done — one binary matrix, two more things to do with it

You now have exactly the kind of presence/absence matrix GMWI2's trained
model consumes. Notebook 04 uses it to compute a simplified health score
by hand; notebook 05 uses it to train and evaluate an actual classifier.

**Next:** `04_health_vs_disease_species.ipynb`.